# Orthogroups
In this notebook, we'll take a look at the output of Orthofinder and identify the photorespiiration enzymes in our target species.

In [1]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from collections import defaultdict
from os import listdir
from os.path import splitext
from tqdm import tqdm

In [17]:
run_date = '12Nov2025'

## Read in data
We'll strt by looking at the `Orthogroups.tsv` file. We also want to import the AT numbers for the Arabidopsis versions of the genes we're looking for.

In [2]:
orthogroups = pd.read_csv('/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/03No2025_all_species_Orthogroups.tsv', sep='\t', low_memory=False)
orthogroups.head()

,Orthogroup,translated_Acoerulea_322_v3.1.cds_primaryTranscriptOnly,translated_Acomosus_321_v3.cds_primaryTranscriptOnly,translated_AgerardiHAP1_783_v1.1.cds_primaryTranscriptOnly,translated_AhypogaeaLine8_886_v1.3.cds_primaryTranscriptOnly,translated_Aoccidentale_449_v0.9.cds_primaryTranscriptOnly,translated_Atequilanavar_WebersBlueHAP1_790_v2.1.cds_primaryTranscriptOnly,translated_Athaliana_447_Araport11.cds_primaryTranscriptOnly,translated_BhybridumBhyb26_693_v2.1.cds_primaryTranscriptOnly,translated_Bplatyphylla_679_v1.1.cds_primaryTranscriptOnly,...,translated_Tlatifoliavar_2019_6_853_v1.1.cds_primaryTranscriptOnly,translated_Tplicata_572_v3.1.cds_primaryTranscriptOnly,translated_Tpratense_385_v2.cds_primaryTranscriptOnly,translated_Vcarteri_317_v2.1.cds_primaryTranscriptOnly,translated_Vdarrowii_700_v1.2.cds_primaryTranscriptOnly,translated_Vvinifera_457_v2.1.cds_primaryTranscriptOnly,translated_YaloifoliaYa24Inoko_839_v2.1.cds_primaryTranscriptOnly,translated_Zmays_833_Zm-B73-REFERENCE-NAM-5.0.55.cds_primaryTranscriptOnly,translated_prepended_CsubellipsoideaC169_227_v2.0.cds_primaryTranscriptOnly,translated_prepended_Smoellendorffii_91_v1.0.cds_primaryTranscriptOnly
0,OG0000000,Aqcoe0541s0001.1 pacid=33091768 polypeptide=Aq...,Aco000135.1 pacid=33042441 polypeptide=Aco0001...,AndgeH1.01AG001000.1 pacid=57338243 polypeptid...,AhLine8.01G009500.1 pacid=64827772 polypeptide...,Anaoc.0001s0502.1 pacid=37553830 polypeptide=A...,AgateH1.01G022100.1 pacid=57831208 polypeptide...,AT1G03510.1 pacid=37392137 polypeptide=AT1G035...,Bhyb26.D01G001400.1 pacid=51520616 polypeptide...,BPChr01G00548 pacid=50813750 polypeptide=BPChr...,...,Tylat.01G005300.1 pacid=62721257 polypeptide=T...,Thupl.28202865s0001.1 pacid=44979454 polypepti...,Tp57577_TGAC_v2_mRNA10243 pacid=35981621 polyp...,NaN,Vadar_g10031.t1 pacid=52046125 polypeptide=Vad...,VIT_200s0179g00220.1 pacid=38049603 polypeptid...,Yucal.01G023700.1 pacid=61526493 polypeptide=Y...,Zm00001eb002930_T001 pacid=61177711 polypeptid...,NaN,Smollendorffii100216 pacid=15420961 polypeptid...
1,OG0000001,Aqcoe0071s0002.1 pacid=33084633 polypeptide=Aq...,Aco000310.1 pacid=33042938 polypeptide=Aco0003...,AndgeH1.01AG103100.1 pacid=57338018 polypeptid...,AhLine8.02G175100.1 pacid=64840868 polypeptide...,Anaoc.0014s0200.1 pacid=37546849 polypeptide=A...,AgateH1.01G048500.1 pacid=57831587 polypeptide...,AT1G04625.1 pacid=37398494 polypeptide=AT1G046...,Bhyb26.D01G037800.1 pacid=51527689 polypeptide...,BPChr01G05174 pacid=50813934 polypeptide=BPChr...,...,NaN,Thupl.28170576s0001.1 pacid=44943991 polypepti...,Tp57577_TGAC_v2_mRNA10862 pacid=35968950 polyp...,NaN,Vadar_g10017.t1 pacid=52046259 polypeptide=Vad...,VIT_215s0021g00810.1 pacid=38040437 polypeptid...,Yucal.01G081800.1 pacid=61527213 polypeptide=Y...,Zm00001eb279530_T001 pacid=61192260 polypeptid...,NaN,NaN
2,OG0000002,Aqcoe0015s0006.1 pacid=33098367 polypeptide=Aq...,Aco002021.1 pacid=33044819 polypeptide=Aco0020...,AndgeH1.01BG234100.1 pacid=57383204 polypeptid...,AhLine8.03G125400.1 pacid=64814314 polypeptide...,Anaoc.0002s0364.1 pacid=37515686 polypeptide=A...,AgateH1.01G077900.1 pacid=57831467 polypeptide...,AT1G42190.1 pacid=37399316 polypeptide=AT1G421...,Bhyb26.D01G535300.1 pacid=51517475 polypeptide...,BPChr04G25220 pacid=50811092 polypeptide=BPChr...,...,Tylat.03G033000.1 pacid=62713494 polypeptide=T...,Thupl.28445139s0001.1 pacid=44951194 polypepti...,Tp57577_TGAC_v2_mRNA11600 pacid=35987994 polyp...,Vocar.0002s0375.1 pacid=32891029 polypeptide=V...,Vadar_g10197.t1 pacid=52044296 polypeptide=Vad...,VIT_200s0227g00030.1 pacid=38050389 polypeptid...,Yucal.01G218700.1 pacid=61530997 polypeptide=Y...,NaN,NaN,Smollendorffii136708 pacid=15419309 polypeptid...
3,OG0000003,Aqcoe5G049000.1 pacid=33086099 polypeptide=Aqc...,Aco021638.1 pacid=33040729 polypeptide=Aco0216...,AndgeH1.01AG074600.1 pacid=57341166 polypeptid...,AhLine8.01G039700.1 pacid=64825845 polypeptide...,Anaoc.0001s0947.1 pacid=37548454 polypeptide=A...,AgateH1.01G03490

In [3]:
with open('/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/sequences/photorespiration_gene_ids.txt') as f:
    photo_genes = [l.strip() for l in f.readlines()]
photo_genes = [g.upper() for g in photo_genes]
photo_genes

['AT5G36700',
 'AT3G14420',
 'AT4G35090',
 'AT1G23310',
 'AT1G70580',
 'AT4G33010',
 'AT2G35370',
 'AT1G11860',
 'AT1G48030',
 'AT4G37930',
 'AT2G13360',
 'AT1G68010',
 'AT1G80380']

## Finding orthogroups
The basic approach here is to find the orthogroups that have the Arabidopsis photorespiration genes. We'll need to do some manual work to confirm them, but as a first step, we can do this in an automated fashion:

In [4]:
# Get the full name of the Arabidopsis column
ara_col = orthogroups.columns[np.where(orthogroups.columns.str.contains('thaliana'))[0]][0]

In [5]:
idx_to_keep = []
for pg in photo_genes:
    contains_df = orthogroups[ara_col].str.contains(pg).dropna()
    idx_to_keep.extend(contains_df[contains_df].index.tolist())
photo_orthos = orthogroups.loc[idx_to_keep].reset_index(drop=True)
photo_orthos

,Orthogroup,translated_Acoerulea_322_v3.1.cds_primaryTranscriptOnly,translated_Acomosus_321_v3.cds_primaryTranscriptOnly,translated_AgerardiHAP1_783_v1.1.cds_primaryTranscriptOnly,translated_AhypogaeaLine8_886_v1.3.cds_primaryTranscriptOnly,translated_Aoccidentale_449_v0.9.cds_primaryTranscriptOnly,translated_Atequilanavar_WebersBlueHAP1_790_v2.1.cds_primaryTranscriptOnly,translated_Athaliana_447_Araport11.cds_primaryTranscriptOnly,translated_BhybridumBhyb26_693_v2.1.cds_primaryTranscriptOnly,translated_Bplatyphylla_679_v1.1.cds_primaryTranscriptOnly,...,translated_Tlatifoliavar_2019_6_853_v1.1.cds_primaryTranscriptOnly,translated_Tplicata_572_v3.1.cds_primaryTranscriptOnly,translated_Tpratense_385_v2.cds_primaryTranscriptOnly,translated_Vcarteri_317_v2.1.cds_primaryTranscriptOnly,translated_Vdarrowii_700_v1.2.cds_primaryTranscriptOnly,translated_Vvinifera_457_v2.1.cds_primaryTranscriptOnly,translated_YaloifoliaYa24Inoko_839_v2.1.cds_primaryTranscriptOnly,translated_Zmays_833_Zm-B73-REFERENCE-NAM-5.0.55.cds_primaryTranscriptOnly,translated_prepended_CsubellipsoideaC169_227_v2.0.cds_primaryTranscriptOnly,translated_prepended_Smoellendorffii_91_v1.0.cds_primaryTranscriptOnly
0,OG0002242,Aqcoe4G129900.1 pacid=33066520 polypeptide=Aqc...,Aco008063.1 pacid=33032299 polypeptide=Aco0080...,AndgeH1.01AG005500.1 pacid=57339218 polypeptid...,AhLine8.04G054000.1 pacid=64839077 polypeptide...,Anaoc.0005s0234.1 pacid=37525864 polypeptide=A...,AgateH1.01G302500.1 pacid=57830997 polypeptide...,AT5G36700.4 pacid=37416686 polypeptide=AT5G367...,Bhyb26.D02G387200.1 pacid=51529138 polypeptide...,BPChr05G14757 pacid=50814855 polypeptide=BPChr...,...,Tylat.09G061000.1 pacid=62733152 polypeptide=T...,Thupl.29379746s0003.1 pacid=44950181 polypepti...,Tp57577_TGAC_v2_mRNA18145 pacid=35961266 polyp...,Vocar.0001s1317.1 pacid=32885481 polypeptide=V...,Vadar_g1901.t1 pacid=52029797 polypeptide=Vada...,VIT_200s0194g00110.1 pacid=38048793 polypeptid...,Yucal.01G302600.1 pacid=61531861 polypeptide=Y...,Zm00001eb064870_T001 pacid=61179706 polypeptid...,Csubellipsoidea61099 pacid=27390732 polypeptid...,Smollendorffii417198 pacid=15401614 polypeptid...
1,OG0000625,Aqcoe1G188500.1 pacid=33077808 polypeptide=Aqc...,Aco005107.1 pacid=33048657 polypeptide=Aco0051...,AndgeH1.01AG054600.1 pacid=57337279 polypeptid...,AhLine8.03G369700.1 pacid=64814523 polypeptide...,Anaoc.0001s2009.1 pacid=37552984 polypeptide=A...,AgateH1.08G075200.1 pacid=57817264 polypeptide...,AT3G14130.1 pacid=37405630 polypeptide=AT3G141...,Bhyb26.D01G048900.1 pacid=51518554 polypeptide...,BPChr04G18347 pacid=50811041 polypeptide=BPChr...,...,Tylat.03G067500.1 pacid=62712103 polypeptide=T...,Thupl.29379579s0003.1 pacid=44979471 polypepti...,Tp57577_TGAC_v2_mRNA30783 pacid=35968999 polyp...,NaN,Vadar_g30372.t1 pacid=52041404 polypeptide=Vad...,VIT_209s0002g00810.7 pacid=38070406 polypeptid...,Yucal.08G073500.1 pacid=61472601 polypeptide=Y...,Zm00001eb070270_T004 pacid=61168717 polypeptid...,NaN,Smollendorffii266585 pacid=15409182 polypeptid...
2,OG0001411,Aqcoe5G121500.1 pacid=33087865 polypeptide=Aqc...,Aco010626.1 pacid=33049307 polypeptide=Aco0106...,AndgeH1.01BG477300.1 pacid=57379424 polypeptid...,AhLine8.08G018300.1 pacid=64830816 polypeptide...,Anaoc.0009s0162.1 pacid=37496986 polypeptide=A...,AgateH1.03G164600.1 pacid=57848818 polypeptide...,AT1G20620.6 pacid=37396564 polypeptide=AT1G206...,Bhyb26.D01G294800.1 pacid=51522680 polypeptide...,BPChr01G16920 pacid=50813365 polypeptide=BPChr...,...,Tylat.01G083900.1 pacid=62718035 polypeptide=T...,Thupl.29379422s0008.1 pacid=44965481 polypepti...,Tp57577_TGAC_v2_mRNA39406 pacid=35955901 polyp...,Vocar.0011s0197.1 pacid=32895313 polypeptide=V...,Vadar_g26040.t1 pacid=52032478 polypeptide=Vad...,VIT_200s0698g00010.2 pacid=38048593 polypeptid...,Yucal.03G162600.1 pacid=61509672 polypeptide=Y...,Zm00001eb002510_T002 pacid=61171381 polypeptid...,Csubellipsoidea15220 pacid=27386617 polypeptid...,Smollendorffii112333 pacid=15412583 polypeptid...
3,

In [6]:
print(f'{len(photo_orthos)} of {len(photo_genes)} photorespiration genes were found.')

13 of 13 photorespiration genes were found.


## Making gene family fastas
We now need to make a fasta containing each gene family across all species. We'll do this by going through each species' fasta a single time, and check whether each sequence belongs to an orthogroup. We'll then append those sequence objects to each gene family to then save out new fastas.

In [13]:
gene_family_lists = defaultdict(list)

base_path = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/base_set'
for species in tqdm(listdir(base_path)):
    species_base = splitext(species)[0]
    fasta_sequences = SeqIO.parse(open(f'{base_path}/{species}'),'fasta')
    for fasta in fasta_sequences:
        desc, sequence = fasta.description, str(fasta.seq)
        # Get the orthogroup of this sequence, if any
        if len(photo_orthos[photo_orthos[species_base] == desc]) > 0:
            # Get orthogroup
            orthogroup = photo_orthos[photo_orthos[species_base] == desc]['Orthogroup'].values[0]
            gene_family_lists[orthogroup].append(fasta)

add_path = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/add_set'
for species in tqdm(listdir(add_path)):
    species_base = splitext(species)[0]
    fasta_sequences = SeqIO.parse(open(f'{add_path}/{species}'),'fasta')
    for fasta in fasta_sequences:
        desc, sequence = fasta.description, str(fasta.seq)
        # Get the orthogroup of this sequence, if any
        if len(photo_orthos[photo_orthos[species_base] == desc]) > 0:
            orthogroup = photo_orthos[photo_orthos[species_base] == desc]['Orthogroup'].values[0]
            gene_family_lists[orthogroup].append(fasta)

100%|██████████| 147/147 [19:30<00:00,  7.96s/it]


In [19]:
outpath = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/gene_family_fastas/'
for orthogroup, seq_list in tqdm(gene_family_lists.items()):
    outfile = f'{outpath}/{orthogroup}_{run_date}.fasta'
    with open(outfile, "w") as output_handle:
        SeqIO.write(seq_list, output_handle, "fasta")

100%|██████████| 12/12 [00:00<00:00, 29.19it/s]
